# Notebook 2 — MAE Fine-tuning Ablation (2CH + 4CH, biplane EF)

Purpose: Produce the **MAE-pretrained rows** of the final ablation table — the *core contribution*.

What this notebook does:
- Loads the MAE-pretrained ViT-Tiny encoder (`mae_pretrained_full.pth`).
- Wraps it in a segmentation model (ViT encoder + conv decoder).
- Fine-tunes on **both 2CH and 4CH views**, at 1%, 5%, 10%, 100% labels, 3 seeds each.
- Reports per-view Dice/HD95 and biplane Simpson's EF MAE.
- Saves the best 10%-label checkpoint for Notebook 3 (KAN head).

Prerequisite: `mae_pretrained_full.pth` in the working directory.

Estimated runtime: ~5–6 h on RTX 3060.

In [1]:
# Cell 1 — Imports, paths, splits
import os, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import SimpleITK as sitk
import cv2
import torchvision.transforms.functional as TF
from scipy.ndimage import label as cc_label
from medpy.metric.binary import hd95
import timm
from timm.models.vision_transformer import Block

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BASE_PATH = 'database/'
RESULTS_DIR = 'results/'
os.makedirs(RESULTS_DIR, exist_ok=True)
MAE_WEIGHTS = 'mae_pretrained_full.pth'
assert os.path.exists(MAE_WEIGHTS), f"Missing: {MAE_WEIGHTS}"

all_patients = sorted([d for d in os.listdir(BASE_PATH)
                       if os.path.isdir(os.path.join(BASE_PATH, d))])
random.seed(42); random.shuffle(all_patients)
train_list, val_list, test_list = all_patients[:400], all_patients[400:450], all_patients[450:500]

Device: cuda


In [2]:
# Cell 2 — Dataset (both views, 224x224 for ViT)
class CamusDatasetFull224(Dataset):
    def __init__(self, patient_list, base_path, target_size=(224, 224), augment=False):
        self.base_path = base_path; self.target_size = target_size; self.augment = augment
        self.samples = []
        for p_id in patient_list:
            for view in ["2CH", "4CH"]:
                for phase in ["ED", "ES"]:
                    self.samples.append({
                        "id": p_id, "view": view, "phase": phase,
                        "img":  f"{p_id}_{view}_{phase}.nii.gz",
                        "mask": f"{p_id}_{view}_{phase}_gt.nii.gz",
                    })

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img  = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(
                    os.path.join(self.base_path, s["id"], s["img"])))).astype(np.float32)
        mask = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(
                    os.path.join(self.base_path, s["id"], s["mask"])))).astype(np.float32)
        mask = (mask == 1).astype(np.float32)
        img  = cv2.resize(img,  self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)
        img  = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img_t  = torch.from_numpy(img).unsqueeze(0).float()
        mask_t = torch.from_numpy(mask).long()
        if self.augment:
            if random.random() > 0.5:
                img_t  = TF.hflip(img_t)
                mask_t = TF.hflip(mask_t.unsqueeze(0)).squeeze(0)
            angle = random.uniform(-15, 15)
            img_t  = TF.rotate(img_t, angle)
            mask_t = TF.rotate(mask_t.unsqueeze(0), angle).squeeze(0)
        return img_t, mask_t, s["view"], s["phase"], s["id"]

train_ds_full = CamusDatasetFull224(train_list, BASE_PATH, augment=True)
val_ds        = CamusDatasetFull224(val_list,   BASE_PATH, augment=False)
test_ds       = CamusDatasetFull224(test_list,  BASE_PATH, augment=False)
print(f"train/val/test = {len(train_ds_full)}/{len(val_ds)}/{len(test_ds)}")

train/val/test = 1600/200/200


In [3]:
# Cell 3 — MAE encoder + segmentation wrapper
class MAEViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=1,
                 embed_dim=192, depth=12, num_heads=3,
                 decoder_embed_dim=128, decoder_depth=4, decoder_num_heads=4,
                 mlp_ratio=4.):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias=True, norm_layer=nn.LayerNorm)
            for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        # decoder kept so load_state_dict matches the file
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim))
        self.decoder_blocks = nn.ModuleList([
            Block(decoder_embed_dim, decoder_num_heads, mlp_ratio, qkv_bias=True, norm_layer=nn.LayerNorm)
            for _ in range(decoder_depth)])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size ** 2 * in_chans, bias=True)

class ViTSegmentationModel(nn.Module):
    def __init__(self, mae_weights_path, n_classes=2):
        super().__init__()
        mae = MAEViT()
        mae.load_state_dict(torch.load(mae_weights_path, map_location='cpu'), strict=False)
        self.patch_embed = mae.patch_embed
        self.cls_token   = mae.cls_token
        self.pos_embed   = mae.pos_embed
        self.blocks      = mae.blocks
        self.norm        = mae.norm
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(192, 128, 2, 2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128,  64, 2, 2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d( 64,  32, 2, 2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d( 32,  16, 2, 2), nn.ReLU(inplace=True),
            nn.Conv2d(16, n_classes, 1),
        )

    def encode_cls(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return x[:, 0, :]   # CLS feature, used by Notebook 3

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        feats = x[:, 1:, :].transpose(1, 2).reshape(x.shape[0], 192, 14, 14)
        return self.decoder(feats)

with torch.no_grad():
    _t = ViTSegmentationModel(MAE_WEIGHTS).to(device)
    print("Output shape sanity:", _t(torch.randn(2,1,224,224).to(device)).shape)
    del _t

C:\Users\najib\AppData\Local\Temp\ipykernel_19444\1727963051.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mae.load_state_dict(torch.load(mae_weights_path, map_locati

Output shape sanity: torch.Size([2, 2, 224, 224])


In [4]:
# Cell 4 — Loss + biplane metrics (same as Notebook 1)
class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__(); self.ce = nn.CrossEntropyLoss()
    def forward(self, pred, target):
        ce = self.ce(pred, target)
        ps = F.softmax(pred, dim=1)
        t_oh = F.one_hot(target, 2).permute(0, 3, 1, 2).float()
        inter = (ps * t_oh).sum((2, 3)); union = (ps + t_oh).sum((2, 3))
        dice = (1 - (2 * inter + 1e-6) / (union + 1e-6)).mean()
        return 0.5 * ce + 0.5 * dice

SPACING = (0.3, 0.3)
def get_largest_cc(mask):
    if mask.sum() == 0: return mask
    lbl, n = cc_label(mask)
    if n == 0: return mask
    counts = np.bincount(lbl.flat)[1:]
    return (lbl == counts.argmax() + 1).astype(np.uint8)

def dice_score(p, t):
    p = (p > 0.5).astype(np.float32); t = (t > 0.5).astype(np.float32)
    return (2 * (p * t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)

def _disk_diameters(mask, spacing, num_disks=20):
    y_idx, _ = np.where(mask > 0.5)
    if len(y_idx) == 0: return None, None
    y_min, y_max = y_idx.min(), y_idx.max(); h_px = y_max - y_min
    if h_px == 0: return None, None
    h_mm = (h_px / num_disks) * spacing[0]
    diams = [int((mask[int(y_min + (i+0.5)*(h_px/num_disks)), :] > 0.5).sum()) * spacing[1]
             for i in range(num_disks)]
    return diams, h_mm

def biplane_simpson_volume(m2, m4, spacing=SPACING, num_disks=20):
    d2, h2 = _disk_diameters(m2, spacing, num_disks)
    d4, h4 = _disk_diameters(m4, spacing, num_disks)
    if d2 is None or d4 is None: return 0.0
    h = 0.5 * (h2 + h4)
    return sum((np.pi/4)*a*b*h for a, b in zip(d2, d4)) / 1000.0

@torch.no_grad()
def evaluate_full(model, dataset, device):
    model.eval()
    dices = {"2CH": [], "4CH": []}; hds = {"2CH": [], "4CH": []}
    per_pat = {}
    for k, s in enumerate(dataset.samples):
        per_pat.setdefault(s["id"], {}).setdefault(s["view"], {})[s["phase"]] = k
    ef_t, ef_p = [], []
    for p_id, views in per_pat.items():
        if not ("2CH" in views and "4CH" in views): continue
        if not all(ph in views[v] for v in ("2CH","4CH") for ph in ("ED","ES")): continue
        gts, preds = {}, {}
        for v in ("2CH","4CH"):
            for ph in ("ED","ES"):
                img, m, _, _, _ = dataset[views[v][ph]]
                pred = torch.argmax(model(img.unsqueeze(0).to(device)), dim=1).cpu().numpy()[0]
                pred = get_largest_cc(pred)
                gts[(v,ph)] = m.numpy(); preds[(v,ph)] = pred
                dices[v].append(dice_score(pred, gts[(v,ph)]))
                if pred.sum() > 0 and gts[(v,ph)].sum() > 0:
                    try: hds[v].append(hd95(pred, gts[(v,ph)]) * SPACING[0])
                    except Exception: pass
        edv_p = biplane_simpson_volume(preds[("2CH","ED")], preds[("4CH","ED")])
        esv_p = biplane_simpson_volume(preds[("2CH","ES")], preds[("4CH","ES")])
        edv_t = biplane_simpson_volume(gts[("2CH","ED")],   gts[("4CH","ED")])
        esv_t = biplane_simpson_volume(gts[("2CH","ES")],   gts[("4CH","ES")])
        if edv_p > 0 and edv_t > 0:
            ef_p.append((edv_p - esv_p)/edv_p * 100)
            ef_t.append((edv_t - esv_t)/edv_t * 100)
    return {
        "dice_2CH":  float(np.mean(dices["2CH"])),
        "dice_4CH":  float(np.mean(dices["4CH"])),
        "dice_mean": float(np.mean(dices["2CH"] + dices["4CH"])),
        "hd95_2CH_mm": float(np.mean(hds["2CH"])) if hds["2CH"] else np.nan,
        "hd95_4CH_mm": float(np.mean(hds["4CH"])) if hds["4CH"] else np.nan,
        "ef_mae_biplane": float(np.mean(np.abs(np.array(ef_t) - np.array(ef_p)))) if ef_p else np.nan,
        "n_patients_ef": len(ef_p),
    }
print("Metrics ready (biplane).")

Metrics ready (biplane).


In [5]:
# Cell 5 — Fine-tuning loop
# OPTIMIZED: LP-FT (linear-probe -> fine-tune) + discriminative LR + AMP + grad-clip
import math  # ADDED: cosine-warmup schedule

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def fraction_indices(n_total, fraction, seed):
    rng = np.random.default_rng(seed)
    k = max(4, int(round(n_total * fraction)))
    return rng.choice(n_total, size=k, replace=False)

def quick_val_dice(model, val_ds, device, batch=8):
    model.eval(); d_acc, n = 0.0, 0
    with torch.no_grad():
        for imgs, msks, *_ in DataLoader(val_ds, batch_size=batch):
            preds = torch.argmax(model(imgs.to(device)), dim=1).cpu().numpy()
            for p, t in zip(preds, msks.numpy()):
                d_acc += dice_score(get_largest_cc(p), t); n += 1
    return d_acc / n

# OPTIMIZED: low-data fine-tuning hyper-parameters (stabilises the 10% regime)
FREEZE_EPOCHS  = 8            # phase 1: train the new decoder only (encoder frozen)
ENC_LR, DEC_LR = 2e-5, 1e-3   # discriminative LRs: tiny for pretrained encoder, large for random decoder
WARMUP         = 5

def finetune(train_loader, val_ds, epochs, lr, device):
    # NOTE: architecture unchanged (ViTSegmentationModel); only the optimisation changes.
    #       `lr` is kept for the call signature but superseded by ENC_LR / DEC_LR.
    model = ViTSegmentationModel(MAE_WEIGHTS, n_classes=2).to(device)
    crit = CombinedLoss()

    # OPTIMIZED: discriminative parameter groups (pretrained encoder vs new decoder)
    enc_params = (list(model.patch_embed.parameters()) + list(model.blocks.parameters())
                  + list(model.norm.parameters()) + [model.cls_token, model.pos_embed])
    dec_params = list(model.decoder.parameters())
    opt = torch.optim.AdamW(
        [{"params": enc_params, "lr": ENC_LR},
         {"params": dec_params, "lr": DEC_LR}], weight_decay=1e-4)

    # OPTIMIZED: LP-FT — freeze the encoder for FREEZE_EPOCHS so the random decoder
    # cannot destroy the MAE-pretrained features early (main cause of low-data collapse).
    for p in enc_params: p.requires_grad = False

    # OPTIMIZED: linear warmup -> cosine decay (scales each group's base LR).
    def lr_factor(ep):
        if ep < WARMUP: return (ep + 1) / WARMUP
        prog = (ep - WARMUP) / max(1, epochs - WARMUP)
        return 0.5 * (1 + math.cos(math.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)

    scaler = torch.cuda.amp.GradScaler()   # ADDED: AMP (mixed precision) for the RTX 3060
    best_d, best_state = -1.0, None
    for ep in range(epochs):
        if ep == FREEZE_EPOCHS:            # phase 2: unfreeze encoder
            for p in enc_params: p.requires_grad = True
        model.train()
        for imgs, msks, *_ in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():                 # ADDED: AMP
                loss = crit(model(imgs), msks)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # ADDED: grad clipping (stability)
            scaler.step(opt); scaler.update()
        sched.step()
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            avg = quick_val_dice(model, val_ds, device)
            if avg > best_d:
                best_d = avg
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ep {ep+1:>3}/{epochs} | val_dice={avg:.4f} | "
                  f"enc_lr={opt.param_groups[0]['lr']:.2e} dec_lr={opt.param_groups[1]['lr']:.2e}")
    model.load_state_dict(best_state)
    return model, best_d

In [6]:
# Cell 6 — MAIN LOOP: fractions x seeds
FRACTIONS = [0.01, 0.05, 0.10, 1.0]
SEEDS     = [42, 123, 7]
EPOCHS    = 50
LR        = 5e-5          # NOTE: superseded by ENC_LR/DEC_LR in finetune(); kept for the call signature
BATCH     = 8

# ADDED: RTX 3060 acceleration knobs
torch.backends.cudnn.benchmark = True     # fixed 224x224 input -> faster convolutions
NUM_WORKERS = 0           # Windows/Jupyter: keep 0 (spawn re-imports the kernel and can hang).
                          # On Linux set 2-4. pin_memory + the cache below are the safe speedups.

# ADDED: cache decoded val/test frames so they are not re-read from .nii.gz every eval
class CachedDataset(torch.utils.data.Dataset):
    def __init__(self, base): self.base, self._cache = base, {}
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        if i not in self._cache: self._cache[i] = self.base[i]
        return self._cache[i]
    @property
    def samples(self): return self.base.samples   # evaluate_full() needs this

if not isinstance(val_ds, CachedDataset):  val_ds  = CachedDataset(val_ds)
if not isinstance(test_ds, CachedDataset): test_ds = CachedDataset(test_ds)

records = []; t0 = time.time()
for frac in FRACTIONS:
    for seed in SEEDS:
        run = f"mae_frac{int(frac*100):03d}_seed{seed}"
        print(f"\n=== {run} ===")
        set_seed(seed)
        if frac < 1.0:
            sub = Subset(train_ds_full, fraction_indices(len(train_ds_full), frac, seed))
        else:
            sub = train_ds_full
        print(f"  train samples: {len(sub)}")
        # OPTIMIZED: pin_memory + (optional) workers for faster host->GPU loading
        loader = DataLoader(sub, batch_size=BATCH, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            persistent_workers=(NUM_WORKERS > 0))
        model, best_val = finetune(loader, val_ds, EPOCHS, LR, device)
        m = evaluate_full(model, test_ds, device)
        m.update({"run": run, "fraction": frac, "seed": seed,
                  "n_train": len(sub), "best_val_dice": best_val})
        records.append(m)
        print(f"  TEST  dice2CH={m['dice_2CH']:.4f}  dice4CH={m['dice_4CH']:.4f}  "
              f"hd95_2CH={m['hd95_2CH_mm']:.2f}mm  hd95_4CH={m['hd95_4CH_mm']:.2f}mm  "
              f"biplane_EF_MAE={m['ef_mae_biplane']:.2f}%")
        pd.DataFrame(records).to_csv(os.path.join(RESULTS_DIR, "mae_finetune.csv"), index=False)
        # Save the 10%-label seed-42 checkpoint for Notebook 3
        if frac == 0.10 and seed == SEEDS[0]:
            torch.save(model.state_dict(),
                       os.path.join(RESULTS_DIR, "mae_finetuned_10pct_seed42.pth"))
            print("  Saved checkpoint for Notebook 3.")
        torch.cuda.empty_cache()   # ADDED: free VRAM between the 12 runs (OOM safety)
print(f"\nTotal runtime: {(time.time()-t0)/60:.1f} min")


=== mae_frac001_seed42 ===
  train samples: 16


C:\Users\najib\AppData\Local\Temp\ipykernel_19444\1727963051.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mae.load_state_dict(torch.load(mae_weights_path, map_locati

  ep   5/50 | val_dice=0.0000 | enc_lr=2.00e-05 dec_lr=1.00e-03
  ep  10/50 | val_dice=0.0000 | enc_lr=1.94e-05 dec_lr=9.70e-04
  ep  15/50 | val_dice=0.0000 | enc_lr=1.77e-05 dec_lr=8.83e-04
  ep  20/50 | val_dice=0.0000 | enc_lr=1.50e-05 dec_lr=7.50e-04
  ep  25/50 | val_dice=0.0000 | enc_lr=1.17e-05 dec_lr=5.87e-04
  ep  30/50 | val_dice=0.0000 | enc_lr=8.26e-06 dec_lr=4.13e-04
  ep  35/50 | val_dice=0.0000 | enc_lr=5.00e-06 dec_lr=2.50e-04
  ep  40/50 | val_dice=0.0000 | enc_lr=2.34e-06 dec_lr=1.17e-04
  ep  45/50 | val_dice=0.0000 | enc_lr=6.03e-07 dec_lr=3.02e-05
  ep  50/50 | val_dice=0.0000 | enc_lr=0.00e+00 dec_lr=0.00e+00
  TEST  dice2CH=0.0000  dice4CH=0.0000  hd95_2CH=nanmm  hd95_4CH=nanmm  biplane_EF_MAE=nan%

=== mae_frac001_seed123 ===
  train samples: 16
  ep   5/50 | val_dice=0.1606 | enc_lr=2.00e-05 dec_lr=1.00e-03
  ep  10/50 | val_dice=0.0000 | enc_lr=1.94e-05 dec_lr=9.70e-04
  ep  15/50 | val_dice=0.0000 | enc_lr=1.77e-05 dec_lr=8.83e-04
  ep  20/50 | val_dice=0.00

In [7]:
# Cell 7 — Aggregate
df = pd.read_csv(os.path.join(RESULTS_DIR, "mae_finetune.csv"))
metrics = ["dice_2CH","dice_4CH","dice_mean","hd95_2CH_mm","hd95_4CH_mm","ef_mae_biplane"]
rows = []
for frac, g in df.groupby("fraction"):
    r = {"fraction": frac}
    for m in metrics:
        r[f"{m}_mean"] = g[m].mean()
        r[f"{m}_std"]  = g[m].std()
    rows.append(r)
agg = pd.DataFrame(rows).sort_values("fraction")
agg.to_csv(os.path.join(RESULTS_DIR, "mae_finetune_summary.csv"), index=False)
print(agg.to_string(index=False))

 fraction  dice_2CH_mean  dice_2CH_std  dice_4CH_mean  dice_4CH_std  dice_mean_mean  dice_mean_std  hd95_2CH_mm_mean  hd95_2CH_mm_std  hd95_4CH_mm_mean  hd95_4CH_mm_std  ef_mae_biplane_mean  ef_mae_biplane_std
     0.01       0.191651      0.331949       0.199344      0.345274        0.195497       0.338611          7.615308              NaN          7.336074              NaN            22.869563                 NaN
     0.05       0.827061      0.008629       0.823956      0.004356        0.825508       0.006221          3.906419         0.159394          4.559698         0.088383            19.724585            1.445535
     0.10       0.861693      0.001805       0.863709      0.001246        0.862701       0.000447          3.089020         0.076223          3.532362         0.024145            12.648985            1.210216
     1.00       0.907349      0.000282       0.914594      0.000819        0.910972       0.000287          1.976739         0.005557          2.038565         